# M04-P01 — VAE vs GAN vs Diffusion

Objetivo: comparar tres familias de generación visual con resultados reconocibles y con la profundidad necesaria para un desarrollador de aplicaciones.

Trabajaremos con **MNIST 28×28**: un problema suficientemente sencillo para entrenar en CPU, pero con mucha más calidad visual que `load_digits` 8×8.

La pregunta principal es: **¿qué aporta cada familia y cuándo tiene sentido entrenar frente a consumir un foundation model?**

In [ ]:
import random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cpu')
DATA_DIR=Path('data/torchvision')
OUT_DIR=Path('outputs/m04_p01'); OUT_DIR.mkdir(parents=True,exist_ok=True)
print('PyTorch:',torch.__version__)
print('Device:',DEVICE)

## 1. Tres ideas generativas

**VAE**
```text
imagen → encoder → espacio latente → decoder → reconstrucción / muestra
```

**GAN**
```text
ruido → Generator → imagen falsa
                    ↕
              Discriminator
```

**Diffusion**
```text
imagen → ruido progresivo
ruido → denoising aprendido paso a paso → imagen
```

Entrenaremos un VAE convolucional y una DCGAN. En Diffusion demostraremos visualmente el proceso de ruido y qué aprende el denoiser; no entrenaremos un diffusion model completo desde cero.

In [ ]:
transform=transforms.ToTensor()
train_ds=datasets.MNIST(DATA_DIR,train=True,download=True,transform=transform)
test_ds=datasets.MNIST(DATA_DIR,train=False,download=True,transform=transform)

TRAIN_LIMIT=30000
g=torch.Generator().manual_seed(SEED)
idx=torch.randperm(len(train_ds),generator=g)[:TRAIN_LIMIT]
train_subset=Subset(train_ds,idx.tolist())
train_loader=DataLoader(train_subset,batch_size=128,shuffle=True,num_workers=0)
test_loader=DataLoader(test_ds,batch_size=256,shuffle=False,num_workers=0)

print('Train usado:',len(train_subset))
print('Test:',len(test_ds))
xb,yb=next(iter(train_loader))
grid=make_grid(xb[:32],nrow=8,padding=2)
plt.figure(figsize=(8,4)); plt.imshow(grid.permute(1,2,0).squeeze(),cmap='gray'); plt.axis('off'); plt.title('Ejemplos reales — MNIST 28×28'); plt.show()

# Parte A — VAE

## 2. VAE a alto nivel

No vamos a inspeccionar `mu`, `logvar` ni cada parámetro. La idea suficiente es:

```text
imagen → encoder → representación latente probabilística → decoder → imagen
```

El VAE intenta reconstruir bien y, al mismo tiempo, mantener un espacio latente suficientemente regular para poder muestrear de él.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self,latent_dim=16):
        super().__init__(); self.latent_dim=latent_dim
        self.encoder=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),nn.ReLU(),
            nn.Conv2d(32,64,4,2,1),nn.ReLU())
        self.fc_mu=nn.Linear(64*7*7,latent_dim)
        self.fc_logvar=nn.Linear(64*7*7,latent_dim)
        self.fc_dec=nn.Linear(latent_dim,64*7*7)
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(),
            nn.ConvTranspose2d(32,1,4,2,1),nn.Sigmoid())
    def encode(self,x):
        h=self.encoder(x).flatten(1); return self.fc_mu(h),self.fc_logvar(h)
    def reparameterize(self,mu,logvar):
        std=torch.exp(.5*logvar); return mu+torch.randn_like(std)*std
    def decode(self,z):
        return self.decoder(self.fc_dec(z).view(-1,64,7,7))
    def forward(self,x):
        mu,lv=self.encode(x); z=self.reparameterize(mu,lv); return self.decode(z),mu,lv

def vae_loss(recon,x,mu,lv,beta=.5):
    rec=F.binary_cross_entropy(recon,x,reduction='sum')/x.size(0)
    kl=-.5*torch.sum(1+lv-mu.pow(2)-lv.exp())/x.size(0)
    return rec+beta*kl,rec,kl

vae=ConvVAE(16).to(DEVICE)
print('Parámetros VAE:',sum(p.numel() for p in vae.parameters()))

In [ ]:
VAE_EPOCHS=5
opt=torch.optim.Adam(vae.parameters(),lr=1e-3)
hist=[]; started=time.perf_counter()
for epoch in range(1,VAE_EPOCHS+1):
    vae.train(); sums=np.zeros(3); seen=0
    for x,_ in train_loader:
        x=x.to(DEVICE); opt.zero_grad(); recon,mu,lv=vae(x)
        loss,rec,kl=vae_loss(recon,x,mu,lv); loss.backward(); opt.step()
        b=x.size(0); sums+=np.array([loss.item(),rec.item(),kl.item()])*b; seen+=b
    row={'epoch':epoch,'loss':sums[0]/seen,'reconstruction':sums[1]/seen,'kl':sums[2]/seen}
    hist.append(row); print(row)
print(f'Tiempo VAE: {time.perf_counter()-started:.1f} s')
display(pd.DataFrame(hist).round(3))

vae.eval(); test_x,test_y=next(iter(test_loader)); test_x=test_x[:16].to(DEVICE)
with torch.no_grad():
    mu,_=vae.encode(test_x); recon=vae.decode(mu)
    samples_vae=vae.decode(torch.randn(16,vae.latent_dim,device=DEVICE))
fig,axes=plt.subplots(3,16,figsize=(16,4.8))
for i in range(16):
    axes[0,i].imshow(test_x[i,0].cpu(),cmap='gray',vmin=0,vmax=1)
    axes[1,i].imshow(recon[i,0].cpu(),cmap='gray',vmin=0,vmax=1)
    axes[2,i].imshow(samples_vae[i,0].cpu(),cmap='gray',vmin=0,vmax=1)
    for r in range(3): axes[r,i].axis('off')
axes[0,0].set_ylabel('Original'); axes[1,0].set_ylabel('Recon'); axes[2,0].set_ylabel('Sample')
plt.suptitle('VAE — originales, reconstrucciones y muestras'); plt.tight_layout(); plt.show()

# Parte B — GAN

## 3. Una GAN más capaz

Entrenaremos una **DCGAN convolucional** sobre imágenes 28×28.

```text
Generator: ruido z → imagen
Discriminator: imagen → real o generada
```

La celda de entrenamiento puede tardar varios minutos en CPU. Siempre usamos el mismo `fixed_noise` al final de cada epoch para poder ver la evolución real del Generator.

In [ ]:
LATENT_GAN=64
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(LATENT_GAN,128*7*7),nn.BatchNorm1d(128*7*7),nn.ReLU(True))
        self.conv=nn.Sequential(
            nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(True),
            nn.ConvTranspose2d(64,1,4,2,1),nn.Tanh())
    def forward(self,z): return self.conv(self.fc(z).view(-1,128,7,7))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,64,4,2,1),nn.LeakyReLU(.2,True),
            nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2,True),
            nn.Flatten(),nn.Linear(128*7*7,1))
    def forward(self,x): return self.net(x).squeeze(1)

def init_weights(m):
    if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d,nn.Linear)):
        nn.init.normal_(m.weight,0,.02)
        if getattr(m,'bias',None) is not None: nn.init.zeros_(m.bias)

G=Generator().to(DEVICE); D=Discriminator().to(DEVICE); G.apply(init_weights); D.apply(init_weights)
print('Generator params:',sum(p.numel() for p in G.parameters()))
print('Discriminator params:',sum(p.numel() for p in D.parameters()))

In [ ]:
GAN_EPOCHS=12
criterion=nn.BCEWithLogitsLoss()
optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(.5,.999))
optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(.5,.999))
fixed_noise=torch.randn(32,LATENT_GAN,device=DEVICE)
history=[]; snapshots={}; started=time.perf_counter()

for epoch in range(1,GAN_EPOCHS+1):
    G.train(); D.train(); gs=ds=0.; batches=0
    for real,_ in train_loader:
        real=real.to(DEVICE)*2-1; b=real.size(0)
        optD.zero_grad()
        real_logits=D(real)
        z=torch.randn(b,LATENT_GAN,device=DEVICE); fake=G(z)
        fake_logits=D(fake.detach())
        dloss=criterion(real_logits,torch.full((b,),.9,device=DEVICE))+criterion(fake_logits,torch.zeros(b,device=DEVICE))
        dloss.backward(); optD.step()

        optG.zero_grad(); glogits=D(fake); gloss=criterion(glogits,torch.ones(b,device=DEVICE)); gloss.backward(); optG.step()
        gs+=gloss.item(); ds+=dloss.item(); batches+=1

    G.eval()
    with torch.no_grad(): snapshots[epoch]=(G(fixed_noise).cpu()+1)/2
    if epoch in {1,3,6,GAN_EPOCHS}: torch.save(G.state_dict(),OUT_DIR/f'generator_epoch_{epoch:02d}.pt')
    row={'epoch':epoch,'g_loss':gs/batches,'d_loss':ds/batches}; history.append(row)
    print(f"epoch {epoch:02d} | G {row['g_loss']:.3f} | D {row['d_loss']:.3f}")
print(f'Tiempo GAN: {time.perf_counter()-started:.1f} s')

In [ ]:
show_epochs=[1,3,6,GAN_EPOCHS]
fig,axes=plt.subplots(len(show_epochs),8,figsize=(10,2.2*len(show_epochs)))
for r,ep in enumerate(show_epochs):
    for c in range(8):
        axes[r,c].imshow(snapshots[ep][c,0],cmap='gray',vmin=0,vmax=1); axes[r,c].axis('off')
    axes[r,0].set_ylabel(f'epoch {ep}')
plt.suptitle('DCGAN — evolución del mismo ruido fijo'); plt.tight_layout(); plt.show()

display(pd.DataFrame(history).round(3))
plt.figure(figsize=(8,4)); plt.plot([r['epoch'] for r in history],[r['g_loss'] for r in history],label='Generator'); plt.plot([r['epoch'] for r in history],[r['d_loss'] for r in history],label='Discriminator'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('GAN — pérdidas'); plt.show()

G.eval()
with torch.no_grad(): gan_final=(G(torch.randn(16,LATENT_GAN,device=DEVICE)).cpu()+1)/2
fig,axes=plt.subplots(2,16,figsize=(16,2.8))
for i in range(16):
    axes[0,i].imshow(samples_vae[i,0].cpu(),cmap='gray',vmin=0,vmax=1); axes[1,i].imshow(gan_final[i,0],cmap='gray',vmin=0,vmax=1)
    axes[0,i].axis('off'); axes[1,i].axis('off')
axes[0,0].set_ylabel('VAE'); axes[1,0].set_ylabel('GAN'); plt.suptitle('Muestras generadas — VAE vs GAN'); plt.tight_layout(); plt.show()

# Parte C — Diffusion

## 4. Demostrar la idea de difusión

No entrenaremos un DDPM completo. Sí ejecutaremos la parte fundamental del proceso.

```text
x₀ → xₜ cada vez más ruidoso → x_T ≈ ruido
```

Durante un entrenamiento real, la red recibe `x_t` y el timestep `t`, e intenta predecir el ruido `epsilon` que se añadió. Durante generación se parte de ruido y se repite el proceso inverso de denoising muchas veces.

In [ ]:
T=1000
betas=torch.linspace(1e-4,.02,T)
alphas=1-betas
alpha_bar=torch.cumprod(alphas,dim=0)

def q_sample(x0,t,noise=None):
    if noise is None: noise=torch.randn_like(x0)
    return alpha_bar[t].sqrt()*x0+(1-alpha_bar[t]).sqrt()*noise,noise

x0=test_x[:4].cpu(); timesteps=[0,50,200,500,750,999]
fig,axes=plt.subplots(4,len(timesteps),figsize=(12,7))
for r in range(4):
    noise=torch.randn_like(x0[r:r+1])
    for c,t in enumerate(timesteps):
        xt,_=q_sample(x0[r:r+1],t,noise)
        axes[r,c].imshow(xt[0,0],cmap='gray'); axes[r,c].axis('off')
        if r==0: axes[r,c].set_title(f't={t}')
plt.suptitle('Forward diffusion — la imagen se convierte progresivamente en ruido'); plt.tight_layout(); plt.show()

demo_x0=test_x[:8].cpu(); demo_t=500; demo_xt,target_epsilon=q_sample(demo_x0,demo_t)
print('x0:',tuple(demo_x0.shape),'x_t:',tuple(demo_xt.shape),'target epsilon:',tuple(target_epsilon.shape),'t:',demo_t)
fig,axes=plt.subplots(2,8,figsize=(12,3))
for i in range(8):
    axes[0,i].imshow(demo_x0[i,0],cmap='gray'); axes[1,i].imshow(demo_xt[i,0],cmap='gray')
    axes[0,i].axis('off'); axes[1,i].axis('off')
axes[0,0].set_ylabel('x0'); axes[1,0].set_ylabel(f'x_t={demo_t}'); plt.suptitle('Entrada que usaría un denoiser durante training'); plt.tight_layout(); plt.show()

## 5. Comparación final y conexión con la aplicación

| Familia | Idea central | Qué hacemos aquí |
|---|---|---|
| VAE | espacio latente probabilístico + decoder | entrenar, reconstruir y muestrear |
| GAN | Generator compite con Discriminator | entrenar DCGAN y observar evolución |
| Diffusion | aprender a invertir ruido progresivo | demostrar noising y objetivo del denoiser |

No incorporaremos esta DCGAN de MNIST a Enterprise GenAI Assistant. Su utilidad es entender qué implica **entrenar un generador propio**.

En las siguientes prácticas volvemos a la aplicación:

```text
M04-P02: Analyze → texto + imagen → análisis multimodal
M04-P03: Create → prompt → foundation image model → activo visual
M04-P04: integración visual en Enterprise GenAI Assistant
```

La comparación importante para un desarrollador es: **entrenar desde cero vs integrar un foundation model ya entrenado**.